## # Données environnementales : quartiers à risque de Libreville

Ce notebook construit un indice de risque par quartier, en combinant :
1. la fréquence d'apparition du quartier dans les épisodes d'inondation
   historiques documentés (chapitre 3.2.2) ;
2. l'altitude du quartier (les zones basses / bas-fonds étant statistiquement
   plus exposées, selon la littérature scientifique locale — Menié Ovono &
   Pottier, 2019).

## ## Étape 1 : Liste des quartiers et coordonnées

Les quartiers ci-dessous ont été identifiés comme récurrents dans les
épisodes d'inondation recensés en section 3.2.2 (presse locale, ReliefWeb,
FloodList). Les coordonnées ont été obtenues via une recherche de lieux
généraliste (Google Places) et constituent une approximation du centre de
chaque quartier, non une donnée cadastrale précise.

In [1]:
import pandas as pd

# Liste des quartiers avec :
# - coordonnées GPS approximatives (latitude, longitude)
# - nombre d'occurrences dans les épisodes d'inondation documentés (section 3.2.2)
quartiers = pd.DataFrame([
    {"quartier": "Nzeng-Ayong",       "latitude": 0.4316, "longitude": 9.4380, "nb_occurrences_inondation": 2},
    {"quartier": "PK8",               "latitude": 0.4080, "longitude": 9.4937, "nb_occurrences_inondation": 2},
    {"quartier": "PK6",               "latitude": 0.4057, "longitude": 9.4798, "nb_occurrences_inondation": 1},
    {"quartier": "Mont-Bouët",        "latitude": 0.3989, "longitude": 9.4528, "nb_occurrences_inondation": 2},
    {"quartier": "Bas-de-Gué-Gué",    "latitude": 0.4356, "longitude": 9.4360, "nb_occurrences_inondation": 1},
    {"quartier": "Oloumi",            "latitude": 0.3790, "longitude": 9.4656, "nb_occurrences_inondation": 1},
    {"quartier": "Plein-Ciel",        "latitude": 0.4020, "longitude": 9.4739, "nb_occurrences_inondation": 1},
    {"quartier": "Awendjé",           "latitude": 0.3864, "longitude": 9.4737, "nb_occurrences_inondation": 1},
    {"quartier": "Batterie 4",        "latitude": 0.4187, "longitude": 9.4280, "nb_occurrences_inondation": 1},
    {"quartier": "Belle-Vue 2",       "latitude": 0.3987, "longitude": 9.4695, "nb_occurrences_inondation": 1},
    {"quartier": "Cité Mebiame",      "latitude": 0.4128, "longitude": 9.4663, "nb_occurrences_inondation": 1},
    {"quartier": "Présidence",        "latitude": 0.3922, "longitude": 9.4419, "nb_occurrences_inondation": 1},
])

print(f"Nombre de quartiers recensés : {len(quartiers)}")
quartiers

Nombre de quartiers recensés : 12


,quartier,latitude,longitude,nb_occurrences_inondation
0,Nzeng-Ayong,0.4316,9.4380,2
1,PK8,0.4080,9.4937,2
2,PK6,0.4057,9.4798,1
3,Mont-Bouët,0.3989,9.4528,2
4,Bas-de-Gué-Gué,0.4356,9.4360,1
5,Oloumi,0.3790,9.4656,1
6,Plein-Ciel,0.4020,9.4739,1
7,Awendjé,0.3864,9.4737,1
8,Batterie 4,0.4187,9.4280,1
9,Belle-Vue 2,0.3987,9.4695,1


## ## Étape 2 : Récupération de l'altitude de chaque quartier

L'altitude est un indicateur indirect de vulnérabilité : les zones basses /
bas-fonds accumulent naturellement les eaux de ruissellement et sont plus
difficiles à drainer, comme le souligne la littérature scientifique sur les
bassins-versants urbains de Libreville. On utilise l'API gratuite
**Open-Elevation** pour obtenir cette donnée automatiquement.

In [2]:
import requests
import time

def recuperer_altitude(latitude, longitude):
    """Interroge l'API Open-Elevation pour une coordonnée donnée."""
    url = "https://api.open-elevation.com/api/v1/lookup"
    params = {"locations": f"{latitude},{longitude}"}
    reponse = requests.get(url, params=params, timeout=30)
    reponse.raise_for_status()
    donnees = reponse.json()
    return donnees["results"][0]["elevation"]

# On récupère l'altitude pour chaque quartier, un par un
altitudes = []
for index, ligne in quartiers.iterrows():
    altitude = recuperer_altitude(ligne["latitude"], ligne["longitude"])
    altitudes.append(altitude)
    print(f"{ligne['quartier']:<20} → {altitude} m")
    time.sleep(1)  # pause pour ne pas surcharger l'API gratuite

quartiers["altitude_m"] = altitudes
quartiers

Nzeng-Ayong          → 12.0 m
PK8                  → 41.0 m
PK6                  → 49.0 m
Mont-Bouët           → 17.0 m
Bas-de-Gué-Gué       → 14.0 m
Oloumi               → 5.0 m
Plein-Ciel           → 21.0 m
Awendjé              → 13.0 m
Batterie 4           → 15.0 m
Belle-Vue 2          → 17.0 m
Cité Mebiame         → 32.0 m
Présidence           → 26.0 m


,quartier,latitude,longitude,nb_occurrences_inondation,altitude_m
0,Nzeng-Ayong,0.4316,9.4380,2,12.0
1,PK8,0.4080,9.4937,2,41.0
2,PK6,0.4057,9.4798,1,49.0
3,Mont-Bouët,0.3989,9.4528,2,17.0
4,Bas-de-Gué-Gué,0.4356,9.4360,1,14.0
5,Oloumi,0.3790,9.4656,1,5.0
6,Plein-Ciel,0.4020,9.4739,1,21.0
7,Awendjé,0.3864,9.4737,1,13.0
8,Batterie 4,0.4187,9.4280,1,15.0
9,Belle-Vue 2,0.3987,9.4695,1,17.0


## ## Étape 3 : Construction d'un indice de risque par quartier

L'indice de risque combine deux facteurs normalisés entre 0 et 1 :
- la fréquence d'occurrence dans les inondations historiques (plus il y a
  d'occurrences, plus le score est élevé) ;
- l'altitude inversée (plus l'altitude est basse, plus le score est élevé),
  pertinente uniquement pour le risque d'inondation par accumulation d'eau
  (elle ne capture pas le risque de glissement de terrain sur pente, qui
  suit une autre logique — voir discussion).

Les deux facteurs sont pondérés à parts égales (50/50) dans cette première
version de l'indice.

In [3]:
# Normalisation de la fréquence d'occurrence entre 0 et 1
occ_min, occ_max = quartiers["nb_occurrences_inondation"].min(), quartiers["nb_occurrences_inondation"].max()
quartiers["score_frequence"] = (quartiers["nb_occurrences_inondation"] - occ_min) / (occ_max - occ_min)

# Normalisation de l'altitude inversée entre 0 et 1
# (1 = altitude la plus basse du groupe = risque le plus élevé)
alt_min, alt_max = quartiers["altitude_m"].min(), quartiers["altitude_m"].max()
quartiers["score_altitude_inversee"] = 1 - (quartiers["altitude_m"] - alt_min) / (alt_max - alt_min)

# Indice de risque combiné (moyenne des deux scores, pondération égale)
quartiers["indice_risque"] = 0.5 * quartiers["score_frequence"] + 0.5 * quartiers["score_altitude_inversee"]

# Tri du tableau par indice de risque décroissant
quartiers_tries = quartiers.sort_values("indice_risque", ascending=False)
quartiers_tries[["quartier", "nb_occurrences_inondation", "altitude_m", "indice_risque"]]

,quartier,nb_occurrences_inondation,altitude_m,indice_risque
0,Nzeng-Ayong,2,12.0,0.920455
3,Mont-Bouët,2,17.0,0.863636
1,PK8,2,41.0,0.590909
5,Oloumi,1,5.0,0.500000
7,Awendjé,1,13.0,0.409091
4,Bas-de-Gué-Gué,1,14.0,0.397727
8,Batterie 4,1,15.0,0.386364
9,Belle-Vue 2,1,17.0,0.363636
6,Plein-Ciel,1,21.0,0.318182
11,Présidence,1,26.0,0.261364


In [4]:
!pip install folium


   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   ---------------------------------------- 2/2 [folium]



## ## Étape 4 : Cartographie interactive des zones à risque

On génère une carte interactive de Libreville, où chaque quartier est
représenté par un cercle dont la taille et la couleur reflètent son indice
de risque d'inondation.

In [5]:
import folium

# Coordonnées approximatives du centre de Libreville
centre_libreville = [0.4162, 9.4673]

# Création de la carte de base
carte = folium.Map(location=centre_libreville, zoom_start=12, tiles="OpenStreetMap")

# Fonction pour choisir une couleur selon le niveau de risque
def couleur_risque(indice):
    if indice >= 0.66:
        return "red"       # risque élevé
    elif indice >= 0.33:
        return "orange"    # risque modéré
    else:
        return "green"     # risque plus faible

# Ajout d'un cercle par quartier
for index, ligne in quartiers.iterrows():
    folium.CircleMarker(
        location=[ligne["latitude"], ligne["longitude"]],
        radius=8 + ligne["indice_risque"] * 20,  # taille proportionnelle au risque
        popup=(f"<b>{ligne['quartier']}</b><br>"
               f"Altitude : {ligne['altitude_m']} m<br>"
               f"Occurrences d'inondation : {ligne['nb_occurrences_inondation']}<br>"
               f"Indice de risque : {ligne['indice_risque']:.2f}"),
        tooltip=ligne["quartier"],
        color=couleur_risque(ligne["indice_risque"]),
        fill=True,
        fill_color=couleur_risque(ligne["indice_risque"]),
        fill_opacity=0.6
    ).add_to(carte)

# Sauvegarde de la carte dans un fichier HTML consultable dans un navigateur
carte.save("carte_risque_inondation_libreville.html")

print("Carte générée avec succès ✅")
carte

Carte générée avec succès ✅


In [6]:
# Sauvegarde de la table des quartiers (avec indice de risque) au format CSV,
# pour réutilisation dans le système d'alerte (notebook 3)
chemin_quartiers = r"C:\Users\ADS\Desktop\Projet_Inondation\quartiers_risque_libreville.csv"
quartiers.to_csv(chemin_quartiers, index=False, encoding="utf-8")
print("Table des quartiers sauvegardée avec succès ✅")

Table des quartiers sauvegardée avec succès ✅
